In [1]:
import pandas as pd
from openai import OpenAI
import os
from src.config import QDRANT_URL, QDRANT_API_KEY, OPENAI_API_KEY, OPENAI_MODEL, OPENAI_API_URL,\
     DENSE_EMBEDDING_MODEL_PATH, OPENAI_MODEL_MINI, INTERIM_DATA_DIR, PROCESSED_DATA_DIR, DEVICE, \
     DEEPSEEK_MODEL
from src.dataset import render_table_ddls, rewrite_query_descriptions_csv, get_sqlite_database_path
from src.script_generator import (
    generate_query_descriptions,
    generate_related_query_descriptions_csv,
    generate_sql_scripts_and_results,
)
from src.vanna_connector import initialize_vanna
from src.search_metrics import run_search_evaluation_pipeline, compute_ranking_metrics
from src.generation_metrics import run_generation_evaluation_pipeline, compute_generation_metrics

/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
url = os.path.join(PROCESSED_DATA_DIR, "sakila", "sakila_inline_short.sqlite.db")
DATABASE_NAME = "sakila"


sqlite_config = {
    "params": {
        "url": str(url) 
    },
    "type": "sqlite"}

qdrant_config = {"fastembed_model": DENSE_EMBEDDING_MODEL_PATH,
                 "url": QDRANT_URL, 
                 "api_key": QDRANT_API_KEY,
                 "device": DEVICE}

openai_config = {"api_key": OPENAI_API_KEY,
                 "model": OPENAI_MODEL,
                 "base_url": OPENAI_API_URL}

In [3]:
vanna_client = initialize_vanna(db_config=sqlite_config,
                                qdrant_config=qdrant_config,
                                openai_config=openai_config)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2322.10it/s]
/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/vanna/legacy/qdrant/qdrant.py:49: UserWarning: Api key is used with an insecure connection.
  self._client = QdrantClient(


# Adding DDL to vector store

In [10]:
table_ddls = render_table_ddls(
    database_name=DATABASE_NAME,
    comment_style="inline",
    comment_variant="short",
)

print(table_ddls[0])

CREATE TABLE act ( -- Актёры фильмов.
  a01 numeric NOT NULL, -- Идентификатор актёра.
  a02 VARCHAR(45) NOT NULL, -- Имя актёра.
  a03 VARCHAR(45) NOT NULL, -- Фамилия актёра.
  a04 TIMESTAMP NOT NULL, -- Дата изменения записи.
  PRIMARY KEY (a01)
);


In [ ]:
# # save as example for inference
# df = pd.DataFrame({"ddl": table_ddls})
# df.to_csv(f"{INTERIM_DATA_DIR}/{DATABASE_NAME}/table_ddls_inline_short.csv", index=False)

In [14]:
for table_ddl in table_ddls:
    vanna_client.train(ddl=table_ddl)

Adding ddl: CREATE TABLE act ( -- Актёры фильмов.
  a01 numeric NOT NULL, -- Идентификатор актёра.
  a02 VARCHAR(45) NOT NULL, -- Имя актёра.
  a03 VARCHAR(45) NOT NULL, -- Фамилия актёра.
  a04 TIMESTAMP NOT NULL, -- Дата изменения записи.
  PRIMARY KEY (a01)
);
Adding ddl: CREATE TABLE cnt ( -- Страны.
  c01 SMALLINT NOT NULL, -- Идентификатор страны.
  c02 VARCHAR(50) NOT NULL, -- Название страны.
  c03 TIMESTAMP, -- Дата изменения записи.
  PRIMARY KEY (c01)
);
Adding ddl: CREATE TABLE cty ( -- Города.
  d01 int NOT NULL, -- Идентификатор города.
  d02 VARCHAR(50) NOT NULL, -- Название города.
  d03 SMALLINT NOT NULL, -- Идентификатор страны.
  d04 TIMESTAMP NOT NULL, -- Дата изменения записи.
  PRIMARY KEY (d01),
  CONSTRAINT fk_cty_cnt FOREIGN KEY (d03) REFERENCES cnt (c01) ON DELETE NO ACTION ON UPDATE CASCADE
);
Adding ddl: CREATE TABLE adr ( -- Адреса.
  e01 int NOT NULL, -- Идентификатор адреса.
  e02 VARCHAR(50) NOT NULL, -- Адрес (строка 1).
  e03 VARCHAR(50) DEFAULT NULL, 

# Generating artificial query description -> SQL queries -> creating descriptions in different format -> adding them to vectore store

In [2]:
openai_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_URL,
)

In [ ]:
descriptions_csv_path, schema_description_path, sqlite_db_path = generate_query_descriptions(
    client=openai_client,
    model=OPENAI_MODEL,
    comment_style="inline",
    comment_variant="short",
    database_name=DATABASE_NAME,
    counts_by_difficulty={"easy": 20, "medium": 20, "hard": 20},
    temperature=1.0,
)

print("Descriptions CSV:", descriptions_csv_path)
print("Schema description:", schema_description_path)
print("SQLite DB:", sqlite_db_path)

Descriptions CSV: /home/user/cursor_projects/vanna-sql/data/interim/sakila/query_descriptions_inline_short.csv
Schema description: /home/user/cursor_projects/vanna-sql/data/interim/sakila/schema_description_inline_short.txt
SQLite DB: /home/user/cursor_projects/vanna-sql/data/processed/sakila/sakila_inline_short.sqlite.db


In [16]:
generation_summary = generate_sql_scripts_and_results(
    client=openai_client,
    model=OPENAI_MODEL,
    sqlite_db_path=sqlite_db_path,
    interim_dir=descriptions_csv_path.parent,
    descriptions_csv_path=descriptions_csv_path,
    schema_description_path=schema_description_path,
    temperature=0.2,
)

generation_summary

{'total': 60,
 'generated': 60,
 'saved': 56,
 'failed': 2,
 'empty': 2,
 'invalid_sql': 0}

In [17]:
# descriptions_csv_path = os.path.join(INTERIM_DATA_DIR, "bank_transaction_monitoring", "query_descriptions_inline_short.csv")
# schema_description_path = os.path.join(INTERIM_DATA_DIR, "bank_transaction_monitoring", "schema_description_inline_short.txt")


rewritten_descriptions_csv_path = rewrite_query_descriptions_csv(
    descriptions_csv_path=descriptions_csv_path,
    client=openai_client,
    model=OPENAI_MODEL_MINI,
    schema_description_path=schema_description_path,
    rewrite_styles=("short", "business", "technical"),
    source_column="query",
    temperature=0.9,
)

print("Rewritten descriptions CSV:", rewritten_descriptions_csv_path)

Rewritten descriptions CSV: /home/user/cursor_projects/vanna-sql/data/interim/sakila/query_descriptions_inline_short_rewritten.csv


In [ ]:
# schema_description_path = os.path.join(INTERIM_DATA_DIR, "bank_transaction_monitoring", "schema_description_inline_short.txt")
# rewritten_descriptions_csv_path = os.path.join(INTERIM_DATA_DIR, "bank_transaction_monitoring", "query_descriptions_inline_short_rewritten.csv")

related_descriptions_csv_path = generate_related_query_descriptions_csv(
    rewritten_descriptions_csv_path=rewritten_descriptions_csv_path,
    schema_description_path=schema_description_path,
    client=openai_client,
    model=DEEPSEEK_MODEL,
    rewrite_styles=("short", "business", "technical"),
    source_column="query",
    temperature=0.9,
    num_queries=3,
)

print("Related descriptions CSV:", related_descriptions_csv_path)

Generating related query descriptions: 100%|██████████| 540/540 [2:14:04<00:00, 14.90s/it]  

Related descriptions CSV: /home/user/cursor_projects/vanna-sql/data/interim/bank_transaction_monitoring/query_descriptions_inline_short_rewritten_related.csv


In [4]:
DATABASE_NAME = "sakila"
related_descriptions_csv_path = '/home/user/cursor_projects/vanna-sql/data/interim/sakila/query_descriptions_inline_short_rewritten_related.csv'

In [ ]:
search_result_paths = run_search_evaluation_pipeline(
    related_descriptions_csv_path=related_descriptions_csv_path,
    interim_dir=INTERIM_DATA_DIR,
    database_name=DATABASE_NAME,
    qdrant_config=qdrant_config,
    openai_config=openai_config,
    db_config=sqlite_config,
    description_styles=("short", "business", "technical"),
    search_query_columns=None,
    n_results=5,
    comment_style="inline",
    comment_variant="short",
)

print("Search results:", search_result_paths)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2710.02it/s]
/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/vanna/legacy/qdrant/qdrant.py:49: UserWarning: Api key is used with an insecure connection.
  self._client = QdrantClient(
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2391.43it/s]
/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/vanna/legacy/qdrant/qdrant.py:49: UserWarning: Api key is used with an insecure connection.
  self._client = QdrantClient(
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4525.91it/s]
/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/vanna/legacy/qdrant/qdrant.py:49: UserWarning: Api key is used with an insecure connection.
  self._client = QdrantClient(
Searching (technical): 100%|██████████| 504/504 [00:14<00:00, 35.81it/s]

Search results: [PosixPath('/home/user/cursor_projects/vanna-sql/data/interim/sakila/search/short_top_5.csv'), PosixPath('/home/user/cursor_projects/vanna-sql/data/interim/sakila/search/business_top_5.csv'), PosixPath('/home/user/cursor_projects/vanna-sql/data/interim/sakila/search/technical_top_5.csv')]
Metrics: [PosixPath('/home/user/cursor_projects/vanna-sql/data/interim/sakila/search/ranking_metrics_short.csv'), PosixPath('/home/user/cursor_projects/vanna-sql/data/interim/sakila/search/ranking_metrics_business.csv'), PosixPath('/home/user/cursor_projects/vanna-sql/data/interim/sakila/search/ranking_metrics_technical.csv')]


In [ ]:
metrics_paths = compute_ranking_metrics(
    search_result_paths=search_result_paths,
    k=5,
)

print("Metrics:", metrics_paths)

In [6]:
for path in metrics_paths:
    print(path.name)
    display(pd.read_csv(path))

ranking_metrics_short.csv


,metric,column,k,value
0,mrr,predicted_short,5,0.535218
1,map_at_k,predicted_short,5,0.535218
2,recall_at_k,predicted_short,5,0.761905
3,mrr,predicted_business,5,0.578571
4,map_at_k,predicted_business,5,0.578571
5,recall_at_k,predicted_business,5,0.773810
6,mrr,predicted_technical,5,0.478175
7,map_at_k,predicted_technical,5,0.478175
8,recall_at_k,predicted_technical,5,0.744048
9,mrr,combined,5,0.562871


ranking_metrics_business.csv


,metric,column,k,value
0,mrr,predicted_short,5,0.516964
1,map_at_k,predicted_short,5,0.516964
2,recall_at_k,predicted_short,5,0.767857
3,mrr,predicted_business,5,0.522421
4,map_at_k,predicted_business,5,0.522421
5,recall_at_k,predicted_business,5,0.773810
6,mrr,predicted_technical,5,0.544940
7,map_at_k,predicted_technical,5,0.544940
8,recall_at_k,predicted_technical,5,0.815476
9,mrr,combined,5,0.546478


ranking_metrics_technical.csv


,metric,column,k,value
0,mrr,predicted_short,5,0.508135
1,map_at_k,predicted_short,5,0.508135
2,recall_at_k,predicted_short,5,0.767857
3,mrr,predicted_business,5,0.544544
4,map_at_k,predicted_business,5,0.544544
5,recall_at_k,predicted_business,5,0.773810
6,mrr,predicted_technical,5,0.498710
7,map_at_k,predicted_technical,5,0.498710
8,recall_at_k,predicted_technical,5,0.785714
9,mrr,combined,5,0.539029


In [ ]:
rewritten_descriptions_csv_path = INTERIM_DATA_DIR / DATABASE_NAME / "query_descriptions_inline_short_rewritten.csv"

predicted_dirs = run_generation_evaluation_pipeline(
    descriptions_csv_path=rewritten_descriptions_csv_path,
    interim_dir=INTERIM_DATA_DIR,
    database_name=DATABASE_NAME,
    models={
        "mini": OPENAI_MODEL_MINI,
    },
    qdrant_config=qdrant_config,
    openai_config=openai_config,
    comment_styles=("inline", ),
    description_styles=("short", "business",),
    db_config=sqlite_config,
    n_folds=5,
)

print("Predicted result dirs:", predicted_dirs)

In [ ]:
ground_truth_results_dir = INTERIM_DATA_DIR / DATABASE_NAME / "results"

generation_metrics_paths = compute_generation_metrics(
    ground_truth_results_dir=ground_truth_results_dir,
    predicted_results_dirs=predicted_dirs,
    descriptions_csv_path=rewritten_descriptions_csv_path,
)

for path in generation_metrics_paths:
    print(path.name)
    display(pd.read_csv(path).label.value_counts())